In [3]:
# 10_label_local_level_clusters.ipynb
#
# For each persona row produced by step 8 (8_cluster_embeddings.ipynb), calls the OpenAI Chat API
# to generate a short title and 2-3 sentence description per cluster,
# then saves enriched results beside the input CSV as <name>_described.csv.
#
# Uses group-level baselines and categorical distributions from step 9
# so the LLM can describe what is *distinctive* about each cluster
# compared to the broader employment group in that LA.
#
# All clusters for a single LA are sent in ONE API call.
#
# Requires OPENAI_API_KEY in environment or .env file.

import sys, os, time, json, shutil
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pathlib import Path
from tqdm import tqdm

import importlib
import data_pipeline.config_variables as _cv_df
importlib.reload(_cv_df)
from data_pipeline.config_variables import DATA_FOLDER
import data_pipeline.config_variables as _cv
_cv.reload_config_variables()

from data_pipeline.config_variables import VARIABLE_MAP, CLUSTER_VARS
from openai import OpenAI

# ── Load .env if present ──────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env', override=False)
    print("Loaded .env")
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. Export it in your shell or add it to .env:\n"
        "  export OPENAI_API_KEY=sk-..."
    )

# ── Config ────────────────────────────────────────────────────────────────────
MODEL          = "gpt-4.1-mini"   # 32 768 output-token limit; cheap & fast
MAX_OUT_TOKENS = 32_000           # hard cap, safely under the model limit
CLUSTER_CSV = Path(f"../{DATA_FOLDER}/8_embedding_clusters/LA_embedding_clusters.csv")
OUTPUT_CSV  = CLUSTER_CSV.parent / (CLUSTER_CSV.stem + "_described.csv")
BASELINES_CSV     = Path(f"../{DATA_FOLDER}/9_group_averages/local_group_baselines.csv")
DISTRIBUTIONS_CSV = Path(f"../{DATA_FOLDER}/9_group_averages/local_group_distributions.csv")
MAX_RETRIES = 3
RETRY_DELAY = 5

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"Model:       {MODEL}")
print(f"Cluster CSV: {CLUSTER_CSV}")
print(f"Output CSV:  {OUTPUT_CSV}")

# ── Load CSVs ─────────────────────────────────────────────────────────────────
df = pd.read_csv(CLUSTER_CSV)
if df.empty:
    raise ValueError(
        f"No rows in {CLUSTER_CSV} — run 8_cluster_embeddings.ipynb (step 8) first."
    )
META_COLS = {'tribe_label', 'size', 'unit_id', 'la_name', 'cluster_level', 'group'}
print(f"Loaded {len(df)} persona rows across {df['unit_id'].nunique()} units")

# Group baselines (from step 9)
df_baselines = pd.read_csv(BASELINES_CSV) if BASELINES_CSV.exists() else pd.DataFrame()
df_dist      = pd.read_csv(DISTRIBUTIONS_CSV) if DISTRIBUTIONS_CSV.exists() else pd.DataFrame()
_uids = set(df["unit_id"].unique())
if not df_baselines.empty:
    df_baselines = df_baselines[df_baselines["unit_id"].isin(_uids)]
if not df_dist.empty:
    df_dist = df_dist[df_dist["unit_id"].isin(_uids)]
if not df_baselines.empty:
    print(f"Loaded {len(df_baselines)} group baselines from step 9")
if not df_dist.empty:
    print(f"Loaded {len(df_dist)} distribution rows from step 9")

# ── Prompts (CLUSTER_VARS + VARIABLE_MAP labels) ─────────────────────────────
_CLUSTERING_LABELS = ", ".join(VARIABLE_MAP[c] for c in CLUSTER_VARS)

SYSTEM_PROMPT = f"""You are a social researcher specialising in UK population demographics.
You will be given the statistical profiles of several population clusters from a single
Local Authority — spanning multiple employment groups (Employed, Retired, Student, etc.) —
derived from the UK Household Longitudinal Study (UKHLS).

You will first see a TOTAL POPULATION BASELINE showing the averages and categorical
breakdowns across ALL people in the LA, regardless of employment group.
Then, for each employment group, you will see a GROUP BASELINE showing the same stats
for that group only. Each cluster's stats follow.
Your job is to identify what makes each cluster distinctive compared to the total
population, its group baseline, and the other clusters.

**Clustering variables** (k-means used ONLY these):
{_CLUSTERING_LABELS}

**Title rules**
- Exactly 3–5 words, vivid and memorable.
- The title must ONLY allude to the clustering variables above.

**Description rules**
The "description" must be a single paragraph explaining what defines this cluster.
Focus on the clustering variables ({_CLUSTERING_LABELS}). Say what distinguishes this
cluster from the total population baseline, its group baseline, and the other clusters.

Respond with a JSON object with a single key "clusters" whose value is an array.
Each array element corresponds to one input cluster and must contain exactly:
  "tribe_label"  — copied verbatim from the input (used to match results back)
  "title"        — per the title rules above
  "description"  — per the description rules above

Respond with valid JSON only — no markdown code fences, no explanation outside the JSON.
Escape any line breaks inside JSON string values as \n."""

print("System prompt OK")


def _format_val(val):
    """Format a value for prompt display."""
    if pd.isna(val):
        return None
    if isinstance(val, float) and val == int(val):
        return str(int(val))
    if isinstance(val, float):
        return f"{val:.1f}"
    return str(val)


def _build_baseline_block(unit_id: str, group: str) -> list[str]:
    """Build the GROUP BASELINE text block for one (unit_id, group)."""
    lines = []
    if df_baselines.empty:
        return lines

    bl = df_baselines[(df_baselines['unit_id'] == unit_id) & (df_baselines['group'] == group)]
    if bl.empty:
        return lines

    bl_row = bl.iloc[0]
    lines.append(f"  === GROUP BASELINE: all {group} in this LA (n={int(bl_row['size']):,}) ===")

    # Continuous/binary stats from the baseline row
    skip = META_COLS | {'tribe_label', 'size', 'unit_id', 'la_name', 'cluster_level', 'group'}
    for col in bl_row.index:
        if col in skip:
            continue
        val = _format_val(bl_row[col])
        if val is not None:
            lines.append(f"    {col}: {val}")

    # Full categorical distributions
    if not df_dist.empty:
        dist = df_dist[(df_dist['unit_id'] == unit_id) & (df_dist['group'] == group)]
        if not dist.empty:
            for var_name, var_dist in dist.groupby('variable', sort=False):
                var_dist = var_dist.sort_values('pct', ascending=False)
                parts = [f"{r['category']}: {r['pct']}%" for _, r in var_dist.iterrows() if r['pct'] >= 1.0]
                if parts:
                    lines.append(f"    {var_name} breakdown: {' | '.join(parts)}")

    lines.append("")
    return lines


def build_la_prompt(unit_id: str, la_name: str, rows: pd.DataFrame) -> str:
    """Build a single prompt containing baselines + all clusters for one LA."""
    lines = [
        f"Local Authority: {la_name} ({unit_id})",
        f"Clusters to label: {len(rows)}",
        "",
    ]

    # Total population baseline for this LA
    lines.extend(_build_baseline_block(unit_id, "Total"))

    # Per-group baselines before the cluster details
    seen_groups = set()
    for _, row in rows.iterrows():
        group = row.get('group', 'Unknown')
        if group not in seen_groups:
            seen_groups.add(group)
            lines.extend(_build_baseline_block(unit_id, group))

    for _, row in rows.iterrows():
        lines += [
            f"--- tribe_label: {row['tribe_label']} ---",
            f"  Employment group: {row.get('group', 'Unknown')}",
            f"  Population size:  {int(row['size']):,}",
        ]
        for col in [c for c in row.index if c not in META_COLS]:
            val = _format_val(row[col])
            if val is not None:
                lines.append(f"  {col}: {val}")
        lines.append("")
    return "\n".join(lines)


# ── One API call per LA ───────────────────────────────────────────────────────
# (unit_id, tribe_label) -> {"gpt_title": ..., "gpt_description": ...}
label_map: dict[tuple, dict] = {}

la_name_col = "la_name" if "la_name" in df.columns else "unit_id"
unit_ids = df["unit_id"].unique().tolist()

_call_num = 0
for unit_id in tqdm(unit_ids, desc="Labelling LAs"):
    la_rows = df[df["unit_id"] == unit_id]
    la_name = la_rows[la_name_col].iloc[0]
    prompt  = build_la_prompt(unit_id, la_name, la_rows)

    _call_num += 1
    if _call_num == 1 or _call_num % 50 == 0:
        print(f"\n{'='*60}\n[Call #{_call_num}] Prompt for {la_name} ({unit_id}):\n{'='*60}")
        print(prompt)
        print('='*60)

    response_text = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
                response_format={"type": "json_object"},
                temperature=0.7,
                max_tokens=min(450 * len(la_rows) + 400, MAX_OUT_TOKENS),
            )
            response_text = resp.choices[0].message.content
            break
        except Exception as e:
            wait = RETRY_DELAY * (2 ** attempt)
            print(f"\n  Error on {unit_id}: {e}  — retrying in {wait}s")
            time.sleep(wait)

    if not response_text:
        continue

    try:
        parsed = json.loads(response_text)
        # Expect {"clusters": [...]} but tolerate a bare list or other wrapper key
        items = parsed.get("clusters") or next(
            (v for v in parsed.values() if isinstance(v, list)), []
        )
        for item in items:
            key = (unit_id, item.get("tribe_label", ""))
            label_map[key] = {
                "gpt_title":       (item.get("title", "") or "").strip() or None,
                "gpt_description": (item.get("description", "") or "").strip() or None,
            }
    except (json.JSONDecodeError, AttributeError) as exc:
        print(f"\n  Parse error for {unit_id}: {exc}")

successful = sum(1 for v in label_map.values() if v["gpt_title"])
print(f"\nCompleted {len(unit_ids)} LA calls — {successful}/{len(df)} clusters labelled")

# ── Merge back and save ───────────────────────────────────────────────────────
df["gpt_title"]       = df.apply(lambda r: label_map.get((r["unit_id"], r["tribe_label"]), {}).get("gpt_title"),       axis=1)
df["gpt_description"] = df.apply(lambda r: label_map.get((r["unit_id"], r["tribe_label"]), {}).get("gpt_description"), axis=1)

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_CSV}")
display(df[["unit_id", "group", "tribe_label", "size", "gpt_title", "gpt_description"]].head(12))

# ── Copy to api/ for deployment ───────────────────────────────────────────────
API_CLUSTERS_DIR = Path('..') / 'api' / 'data' / 'clusters'
API_CLUSTERS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(OUTPUT_CSV, API_CLUSTERS_DIR / OUTPUT_CSV.name)
print(f'Copied {OUTPUT_CSV.name}  ->  api/data/clusters/')


Loaded .env
Model:       gpt-4.1-mini
Cluster CSV: ../data/9_cluster_local_level/LA_london_clusters.csv
Output CSV:  ../data/9_cluster_local_level/LA_london_clusters_described.csv
Loaded 65 cluster rows across 4 units [USE_FOUR_LA_SUBSET=True]
Loaded 24 group baselines from step 9
Loaded 984 distribution rows from step 9
System prompt OK


Labelling LAs:   0%|          | 0/4 [00:00<?, ?it/s]


[Call #1] Prompt for Hounslow (E09000018):
Local Authority: Hounslow (E09000018)
Clusters to label: 12

  === GROUP BASELINE: all Employed in this LA (n=84,192) ===
    Age: 47.5
    Sex (Derived): Male
    Sex (Derived) %: 56
    Ethnic group: White
    Ethnic group %: 58
    Highest qualification: 2.5
    Employment status: Employed: 100%
    Marital status: Married/Civil partner
    Marital status %: 55
    Housing tenure (Own/Rent): Owner-occupied
    Housing tenure (Own/Rent) %: 67
    Composition of household (LFS): Couple, 2 children
    Composition of household (LFS) %: 17
    Self-rated general health: Good
    Self-rated general health %: 38
    Sex (Derived) breakdown: Male: 55.6% | Female: 44.4%
    Ethnic group breakdown: White: 57.6% | Pakistani / Bangladeshi: 15.4% | Indian: 14.1% | Other Asian: 3.7% | Mixed: 2.8% | African: 2.0% | Caribbean: 1.9% | Arab: 1.9%
    Employment status breakdown: Employed: 100.0%
    Marital status breakdown: Married/Civil partner: 54.9% | 

Labelling LAs: 100%|██████████| 4/4 [02:04<00:00, 31.16s/it]


Completed 4 LA calls — 65/65 clusters labelled
Saved 65 rows to ../data/9_cluster_local_level/LA_london_clusters_described.csv


,unit_id,group,tribe_label,size,gpt_title,gpt_description
0,E09000018,Employed,Employed 1,46149,Mature Married Homeowners,This cluster consists of older employed males ...
1,E09000018,Employed,Employed 2,38043,Single Female Renters,This cluster is characterized by younger emplo...
2,E09000018,Retired,Retired 1,11468,Married Retired Male Couples,This retired cluster is mainly males (60%) age...
3,E09000018,Retired,Retired 2,10887,Widowed Older Female Renters,This cluster is older retired females (66%) ag...
4,E09000018,Retired,Retired 3,548,Small White Retired Couples,A small cluster of retired males (57%) aged 74...
5,E09000018,Unemployed,Unemployed 1,4935,Single Younger White Renters,This unemployed cluster is younger males (53% ...
6,E09000018,Unemployed,Unemployed 2,3103,Married Older South Asian Renters,This cluster is older unemployed females (69%)...
7,E09000018,Student,Student,3142,Young Single Students,This cluster encompasses young students (age 2...
8,E09000018,On leave,On leave 1,4634,Married Female Homeowners,This cluster consists mostly of females (93%) ...
9,E09000018,On leave,On leave 2,86,Single Caribbean Female Renters,"A small cluster of females (87%) aged 48.6, pr..."


Copied LA_london_clusters_described.csv  ->  api/data/clusters/
